# Spotimind — 01: embed the catalog with TRIBE v2 (audio-only, Colab T4)

Produces `data/` exactly as specified in `docs/DATA_CONTRACT.md`. Settings come from Phase 0 (`00_phase0_feasibility.ipynb`).

**Steps**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run **[I] Install**. It restarts the runtime on purpose (kernel dies once). Then continue from **[C1]**.
3. Mount Drive: either let **[C1]**'s assert tell you, or use the **Files sidebar → Mount Drive** button (more reliable than the `drive.mount()` popup).
4. Put the mp3s in `MyDrive/brain-mind/` (or edit `AUDIO_DIR`). Outputs go to `MyDrive/brain-mind-output/`.
5. **[C5]** runs the catalog in a background thread, with one checkpoint per song in `work/`. If Colab disconnects, rerun everything: finished songs are skipped. To add songs later, drop the mp3s in the folder and rerun.
6. **[C7]** writes `data/`, and **[C8]** prints a summary and zips `data/` into `spotimind_data.zip`.

**No Hugging Face token needed**: audio-only inference never loads the gated LLaMA text model.

Measured on a T4: ~25 s per 3–4 min song (incl. Drive I/O), peak GPU ~7–8 GB.


In [ ]:
# [I] Install TRIBE v2 (pinned commit). tribev2 pins numpy==2.2.6 and torch 2.6; Colab's torchaudio must be forced to match.
# The runtime restarts at the end of this cell (expected). Then continue with [C1].
!uv pip install --system -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git@af58661791a351a448a489042a28f6c37e1c14b7"
!uv pip install --system -q "torchaudio==2.6.0" "torch==2.6.0" "numpy==2.2.6" mutagen
!pip list 2>/dev/null | grep -iE "^(torch|torchaudio|torchvision|numpy|transformers|neuralset|tribev2|nilearn|mutagen) "
import os; os.kill(os.getpid(), 9)  # restart runtime

In [ ]:
# [C0] Mount Google Drive (if this popup fails, use the Files sidebar > Mount Drive button instead)
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive", timeout_ms=300000)

In [ ]:
# [C1] Config
import os, re, json, time, subprocess, unicodedata, traceback
from pathlib import Path
import numpy as np, pandas as pd, torch

DRIVE      = Path("/content/drive/MyDrive")
AUDIO_DIR  = DRIVE / "brain-mind"                 # the user's mp3s
OUT_DIR    = DRIVE / "brain-mind-output"          # resumable per-song work + final data/
WORK_DIR   = OUT_DIR / "work"                     # one .npz per processed song (checkpoint)
DATA_DIR   = OUT_DIR / "data"                     # final files per docs/DATA_CONTRACT.md
LOCAL_TMP  = Path("/content/bf_tmp"); CACHE = Path("/content/bf_cache")

MAX_SECONDS    = 300      # full songs, capped
WINDOW_MODE    = False    # True -> use WINDOW_SECONDS from the middle of the song instead
WINDOW_SECONDS = 90
TRIM_START_S   = 3        # Phase 0: first ~2 s of any input are edge-affected
TRIM_END_S     = 2        # Phase 0: last ~1-2 s are edge-affected
N_BINS         = 16
BLOCK_WEIGHTS  = {"mean": 1.0, "std": 1.0, "shape": 1.0}
AUDITORY_MIN_FRACTION = 0.30
TRIBE_COMMIT   = "af58661791a351a448a489042a28f6c37e1c14b7"

for d in (WORK_DIR, DATA_DIR / "timelines", LOCAL_TMP, CACHE): d.mkdir(parents=True, exist_ok=True)
assert os.path.ismount("/content/drive"), "Drive not mounted: use the Files sidebar > Mount Drive"
AUDIO_EXT = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
audio_files = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXT)
print(f"{len(audio_files)} audio files in {AUDIO_DIR} | already processed: {len(list(WORK_DIR.glob('*.npz')))}")
print("torch", torch.__version__, "| numpy", np.__version__, "| GPU", torch.cuda.get_device_name(0))
assert np.__version__ == "2.2.6" and torch.__version__.startswith("2.6"), "run the install cell first"

In [ ]:
# [C2] Song metadata: stable ids, title/artist (ID3 first, then "Artist - Title" filename), duration
from mutagen import File as MutagenFile

def slugify(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")[:80] or "song"

def clean_stem(stem):
    return re.sub(r"\s*\((spotisaver|official.*?|audio|lyrics?)\)\s*$", "", stem, flags=re.I).strip()

def read_meta(path):
    title = artist = None
    try:
        tags = MutagenFile(path, easy=True)
        if tags is not None and tags.tags:
            title = (tags.tags.get("title") or [None])[0]; artist = (tags.tags.get("artist") or [None])[0]
    except Exception:
        pass
    stem = clean_stem(path.stem)
    if not title or not artist:
        if " - " in stem:
            a, t = stem.split(" - ", 1); artist = artist or a.strip(); title = title or t.strip()
        else:
            title = title or stem
    return title, artist

def duration_s(path):
    return float(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(path)]))

songs_meta, seen = [], set()
for p in audio_files:
    sid = slugify(clean_stem(p.stem)); base, k = sid, 2
    while sid in seen: sid = f"{base}-{k}"; k += 1
    seen.add(sid)
    title, artist = read_meta(p)
    songs_meta.append({"id": sid, "title": title, "artist": artist, "filename": p.name, "path": str(p)})
print(len(songs_meta), "songs; sample:")
for m in songs_meta[:4]: print("  ", m["id"], "|", m["title"], "|", m["artist"])
assert len({m["id"] for m in songs_meta}) == len(songs_meta)

In [ ]:
# [C3] Atlas: Schaefer-400 (7 networks, native fsaverage5 .annot from CBIG) + Destrieux auditory carve-out
# (scripts/atlas.py holds the same logic for local use; keep them in sync)
import nibabel as nib, urllib.request
from nilearn import datasets
SCH = "Schaefer2018_400Parcels_7Networks_order"
BASE = ("https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/stable_projects/brain_parcellation/"
        "Schaefer2018_LocalGlobal/Parcellations/FreeSurfer5.3/fsaverage5/label")
vert_parcel, parcel_names = [], []
for h, off in (("lh", 0), ("rh", 200)):
    f = LOCAL_TMP / f"{h}.{SCH}.annot"
    if not f.exists(): urllib.request.urlretrieve(f"{BASE}/{h}.{SCH}.annot", f)
    lab, ctab, names = nib.freesurfer.read_annot(str(f))
    names = [n.decode() for n in names]
    assert lab.shape[0] == 10242 and len(names) == 201, (lab.shape, len(names))
    # label 0 = medial wall/background, 1..200 = parcels in Schaefer order
    vert_parcel.append(np.where(lab > 0, lab - 1 + off, -1)); parcel_names += names[1:]
vert_parcel = np.concatenate(vert_parcel).astype(np.int16)        # [20484], -1 = medial wall
P = len(parcel_names); assert P == 400 and vert_parcel.max() == 399
assert all("_LH_" in n for n in parcel_names[:200]) and all("_RH_" in n for n in parcel_names[200:])

# vertex -> parcel averaging matrix [20484, P] (column-normalised), so parcel timeline = preds @ AVG
counts = np.bincount(vert_parcel[vert_parcel >= 0], minlength=P)
AVG = np.zeros((vert_parcel.size, P), np.float32)
idx = np.where(vert_parcel >= 0)[0]; AVG[idx, vert_parcel[idx]] = 1.0 / counts[vert_parcel[idx]]
print(f"parcels: {P}, vertices per parcel min/median/max = {counts.min()}/{int(np.median(counts))}/{counts.max()}, medial wall = {(vert_parcel<0).sum()}")

# networks
YEO = {"Vis": "visual", "SomMot": "somatomotor", "DorsAttn": "dorsal_attention", "SalVentAttn": "ventral_attention",
       "Limbic": "limbic", "Cont": "control", "Default": "default"}
parcel_net = [YEO[n.split("_")[2]] for n in parcel_names]
des = datasets.fetch_atlas_surf_destrieux()
dlabels = [l.decode() if isinstance(l, bytes) else str(l) for l in des["labels"]]
dmap = np.concatenate([des["map_left"], des["map_right"]]).astype(int)
AUD_DESTRIEUX = ["G_temp_sup-G_T_transv", "S_temporal_transverse", "G_temp_sup-Plan_tempo", "G_temp_sup-Lateral"]
aud_vert = np.isin(dmap, [dlabels.index(n) for n in AUD_DESTRIEUX])
aud_frac = np.array([aud_vert[vert_parcel == p].mean() for p in range(P)])
would_qualify = {}
for p in range(P):
    if aud_frac[p] >= AUDITORY_MIN_FRACTION:
        would_qualify.setdefault(parcel_net[p], []).append(parcel_names[p])
        if parcel_net[p] == "somatomotor": parcel_net[p] = "auditory"
NET_ORDER = ["auditory", "visual", "somatomotor", "dorsal_attention", "ventral_attention", "limbic", "control", "default"]
net_parcels = {n: [p for p in range(P) if parcel_net[p] == n] for n in NET_ORDER}
for n in NET_ORDER: print(f"  {n:18s} {len(net_parcels[n]):3d} parcels")
print("parcels >= 30% Destrieux-auditory, by original Yeo network:", {k: len(v) for k, v in would_qualify.items()})
assert len(net_parcels["auditory"]) >= 4, "auditory carve-out found too few parcels"
NETMAT = np.zeros((P, len(NET_ORDER)), np.float32)     # parcel -> network mean
for j, n in enumerate(NET_ORDER): NETMAT[net_parcels[n], j] = 1.0 / len(net_parcels[n])

In [ ]:
# [C4] Model + per-song processing (audio-only path verified in Phase 0). Writes WORK_DIR/{id}.npz as a checkpoint.
# NOTE: model.get_events_dataframe(audio_path=...) would run WhisperX + the gated LLaMA text branch; we bypass it.
from tribev2.demo_utils import TribeModel, get_audio_and_text_events
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=str(CACHE), config_update={"data.num_workers": 2})
AF = model.data.audio_feature  # TRIBE's Wav2VecBert extractor (features cached on disk after predict)

def process_song(m):
    src = Path(m["path"]); dur = duration_s(src)
    if WINDOW_MODE and dur > WINDOW_SECONDS:
        clip_start, clip_len = max(0.0, dur / 2 - WINDOW_SECONDS / 2), float(WINDOW_SECONDS)
    else:
        clip_start, clip_len = 0.0, min(dur, float(MAX_SECONDS))
    wav = LOCAL_TMP / f"{m['id']}.wav"
    subprocess.run(["ffmpeg","-y","-v","error","-ss",str(clip_start),"-i",str(src),"-t",str(clip_len),
                    "-ac","1","-ar","16000",str(wav)], check=True)
    clip_len = duration_s(wav)
    ev = get_audio_and_text_events(pd.DataFrame([{"type":"Audio","filepath":str(wav),"start":0,
                                                   "timeline":"default","subject":"default"}]), audio_only=True)
    assert set(ev.type.unique()) == {"Audio"}
    preds, segs = model.predict(events=ev, verbose=False)                  # [T_raw, 20484], 1 Hz
    starts = np.array([s.start for s in segs]); assert np.allclose(np.diff(starts), 1.0), "non-contiguous segments"
    assert preds.shape[1] == vert_parcel.size and np.isfinite(preds).all()
    T_raw = preds.shape[0]
    keep = slice(TRIM_START_S, T_raw - TRIM_END_S); T = keep.stop - keep.start
    assert T >= N_BINS, f"too short after trim: {T}s"
    timeline = (preds[keep] @ AVG).astype(np.float32)                    # [T, 400]
    feats = AF(ev, start=0.0, duration=clip_len)                          # [2, 1024, T@2Hz]
    feats = (feats.detach().cpu().numpy() if hasattr(feats, "detach") else np.asarray(feats)).astype(np.float32)
    f0, f1 = 2 * TRIM_START_S, min(feats.shape[-1], 2 * (T_raw - TRIM_END_S))
    audio_vec = feats[..., f0:f1].mean(-1).reshape(-1)                    # [2048]
    rec = dict(m, duration_s=round(dur, 3), clip_start_s=round(clip_start, 3), clip_len_s=round(clip_len, 3),
               raw_rows=int(T_raw), bin_count=int(T), processed_duration_s=float(T),
               timeline_start_s=round(clip_start + TRIM_START_S, 3))
    tmp = WORK_DIR / f"{m['id']}.tmp.npz"
    np.savez(tmp, timeline=timeline, audio_vec=audio_vec, meta=json.dumps(rec, ensure_ascii=False))
    tmp.rename(WORK_DIR / f"{m['id']}.npz")                               # atomic-ish: only complete files count
    wav.unlink(missing_ok=True)
    return rec, timeline, audio_vec

# smoke test on the first song (skipped if already processed)
if not (WORK_DIR / f"{songs_meta[0]['id']}.npz").exists():
    t0 = time.time(); rec, tl, av = process_song(songs_meta[0])
    print(f"{rec['id']}: {rec['duration_s']}s -> timeline {tl.shape}, audio_vec {av.shape}, start {rec['timeline_start_s']}s in {time.time()-t0:.1f}s")
    print("auditory vs visual network mean:", (tl @ NETMAT)[:, [0, 1]].mean(0).round(3), "| peak GPU", round(torch.cuda.max_memory_allocated()/1e9, 2), "GB")

In [ ]:
# [C5] Process the whole catalog (resumable: skips songs with an existing WORK_DIR/{id}.npz).
# Runs in a background thread so progress can be polled with [C6]; rerun this cell after a disconnect.
import threading, logging
for name in ("tribev2", "tribev2.main", "tribev2.demo_utils", "neuralset", "huggingface_hub"):
    logging.getLogger(name).setLevel(logging.ERROR)
import warnings; warnings.filterwarnings("ignore")
LOG = OUT_DIR / "run_log.jsonl"
STATE = {"done": 0, "skipped": 0, "failed": [], "running": True, "current": None, "t0": time.time(), "compute_s": 0.0}

def run_catalog():
    try:
        for m in songs_meta:
            if (WORK_DIR / f"{m['id']}.npz").exists(): STATE["skipped"] += 1; continue
            STATE["current"] = m["id"]; t0 = time.time()
            try:
                rec, _, _ = process_song(m); el = time.time() - t0; STATE["compute_s"] += el; STATE["done"] += 1
                entry = {"id": m["id"], "ok": True, "seconds": round(el, 1), "audio_s": rec["clip_len_s"]}
            except Exception as e:
                STATE["failed"].append({"filename": m["filename"], "error": repr(e)[:300]})
                entry = {"id": m["id"], "ok": False, "error": repr(e)[:300], "tb": traceback.format_exc()[-800:]}
            with open(LOG, "a") as f: f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            torch.cuda.empty_cache()
    finally:
        STATE["running"] = False; STATE["current"] = None

worker = threading.Thread(target=run_catalog, daemon=True); worker.start()
print("started; poll with [C6]")

In [ ]:
# [C6] Progress (rerun as often as you like)
n_done = len(list(WORK_DIR.glob("*.npz"))); el = time.time() - STATE["t0"]
rate = STATE["compute_s"] / max(STATE["done"], 1)
print(f"running={STATE['running']} | checkpoints {n_done}/{len(songs_meta)} | this run: done {STATE['done']}, skipped {STATE['skipped']}, "
      f"failed {len(STATE['failed'])} | {el/60:.1f} min elapsed | ~{rate:.0f}s/song | ETA {rate*(len(songs_meta)-n_done)/60:.0f} min | now: {STATE['current']}")
for f in STATE["failed"][-5:]: print("  FAILED", f)

In [ ]:
# [C6b] Block until the catalog run finishes (optional; convenient when driving the notebook remotely)
worker.join()
print(f"finished: done {STATE['done']}, skipped {STATE['skipped']}, failed {len(STATE['failed'])}, "
      f"{(time.time()-STATE['t0'])/60:.1f} min")

In [ ]:
# [C7] Assemble data/ (docs/DATA_CONTRACT.md): songs.json, timelines/, embeddings, networks.json, meta.json
# Embedding functions are an exact copy of backend/app/embedding.py.
import datetime as dt, shutil
assert not STATE.get("running"), "catalog run still in progress; wait for [C6] to show running=False"
EPS = 1e-6
def time_bins(x, n_bins): return np.stack([c.mean(0) for c in np.array_split(x, n_bins, axis=0)])
def zscore_columns(x): return (x - x.mean(0)) / np.maximum(x.std(0), EPS)
def l2_rows(x): return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), EPS)
def brain_embeddings(timelines, netmat, n_bins, weights):
    per = [{"mean": t.mean(0), "std": t.std(0), "shape": time_bins(t @ netmat, n_bins).reshape(-1)} for t in timelines]
    blocks, info = [], []
    for name in ("mean", "std", "shape"):
        x = np.stack([s[name] for s in per]).astype(np.float64)
        blocks.append(weights[name] * l2_rows(zscore_columns(x))); info.append({"name": name, "dim": int(x.shape[1]), "weight": float(weights[name])})
    return l2_rows(np.concatenate(blocks, 1)).astype(np.float32), info
def audio_embeddings(vecs): return l2_rows(zscore_columns(np.stack(vecs).astype(np.float64))).astype(np.float32)

songs, timelines, audio_vecs = [], [], []
for m in songs_meta:
    f = WORK_DIR / f"{m['id']}.npz"
    if not f.exists(): continue
    z = np.load(f); rec = json.loads(str(z["meta"]))
    songs.append({k: rec[k] for k in ("id","title","artist","filename","duration_s","processed_duration_s","bin_count","timeline_start_s")})
    timelines.append(z["timeline"].astype(np.float32)); audio_vecs.append(z["audio_vec"].astype(np.float32))
assert all(t.shape[1] == P and np.isfinite(t).all() for t in timelines)
emb_brain, blocks = brain_embeddings(timelines, NETMAT, N_BINS, BLOCK_WEIGHTS)
emb_audio = audio_embeddings(audio_vecs)

tl_dir = DATA_DIR / "timelines"
for old in tl_dir.glob("*.npy"):
    if old.stem not in {s["id"] for s in songs}: old.unlink()     # songs removed from the folder
for s, t in zip(songs, timelines): np.save(tl_dir / f"{s['id']}.npy", t)
np.save(DATA_DIR / "embeddings_brain.npy", emb_brain); np.save(DATA_DIR / "embeddings_audio.npy", emb_audio)
(DATA_DIR / "songs.json").write_text(json.dumps(songs, indent=1, ensure_ascii=False))

counts = np.bincount(vert_parcel[vert_parcel >= 0], minlength=P)
NET_INFO = {
 "auditory": ("Auditory", None, "Schaefer SomMot parcels overlapping Destrieux auditory cortex (Heschl's gyrus, transverse sulcus, planum temporale, lateral superior temporal gyrus)."),
 "visual": ("Visual", "Vis", "Yeo-7 visual network."),
 "somatomotor": ("Somatomotor", "SomMot", "Yeo-7 somatomotor network, minus the parcels moved to Auditory."),
 "dorsal_attention": ("Dorsal attention", "DorsAttn", "Yeo-7 dorsal attention network (goal-directed attention)."),
 "ventral_attention": ("Salience / ventral attention", "SalVentAttn", "Yeo-7 salience / ventral attention network."),
 "limbic": ("Limbic", "Limbic", "Yeo-7 limbic network (orbitofrontal cortex, temporal pole). Low fMRI signal quality in these areas."),
 "control": ("Control", "Cont", "Yeo-7 frontoparietal control network."),
 "default": ("Default mode", "Default", "Yeo-7 default mode network."),
}
kept_elsewhere = {k: len(v) for k, v in would_qualify.items() if k != "somatomotor"}
networks_doc = {
 "atlas": f"{SCH} (fsaverage5, CBIG)", "auditory_source": "Destrieux 2009 (nilearn fetch_atlas_surf_destrieux, fsaverage5)",
 "n_parcels": P,
 "networks": [{"id": n, "name": NET_INFO[n][0], "yeo": NET_INFO[n][1], "description": NET_INFO[n][2], "parcels": net_parcels[n]} for n in NET_ORDER],
 "parcels": [{"index": p, "name": parcel_names[p], "hemi": "L" if p < 200 else "R", "network": parcel_net[p], "n_vertices": int(counts[p])} for p in range(P)],
 "assumptions": [
  "Parcellation: Schaefer 2018, 400 parcels, 7-network order, native FreeSurfer fsaverage5 annotation (CBIG). No volume-to-surface projection.",
  "TRIBE v2 output vertex order assumed to be [left 10242, right 10242] fsaverage5; verified in Phase 0 by left/right regional symmetry (r=0.97).",
  "Each parcel's network comes from its Schaefer name (Yeo 2011, 7 networks).",
  f"Auditory: a SomMot parcel is relabelled 'auditory' when >= {int(AUDITORY_MIN_FRACTION*100)}% of its vertices fall in Destrieux labels {', '.join(AUD_DESTRIEUX)}.",
  f"Parcels in other Yeo networks that also meet the auditory overlap rule are left in their Yeo network: {kept_elsewhere or 'none'}.",
  "No language network: Yeo-7 has none, so it is omitted rather than approximated.",
  "A network value is the plain mean of its parcels' values; a parcel value is the plain mean of its vertices' TRIBE predictions.",
 ]}
(DATA_DIR / "networks.json").write_text(json.dumps(networks_doc, indent=1))

failures = [json.loads(l) for l in open(LOG)] if LOG.exists() else []
failures = [{"filename": next((m["filename"] for m in songs_meta if m["id"] == e["id"]), e["id"]), "error": e["error"]}
            for e in failures if not e["ok"] and not (WORK_DIR / f"{e['id']}.npz").exists()]
meta = {
 "mock": False, "created_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
 "tribe": {"repo": "facebookresearch/tribev2", "commit": TRIBE_COMMIT, "weights": "facebook/tribev2"},
 "mesh_space": "fsaverage5", "parcellation": SCH, "n_parcels": P, "tr_s": 1.0,
 "max_seconds": MAX_SECONDS, "window_mode": WINDOW_MODE, "window_seconds": WINDOW_SECONDS,
 "trim_start_s": TRIM_START_S, "trim_end_s": TRIM_END_S, "n_bins": N_BINS,
 "brain_embedding": {"blocks": blocks, "dim": int(emb_brain.shape[1]), "normalization": "catalog z-score per feature, L2 per block, weighted concat, L2 row"},
 "audio_embedding": {"source": "tribev2 Wav2VecBert extractor (facebook/w2v-bert-2.0), layer groups 0.5/0.75/1.0, mean over trimmed span",
                     "dim": int(emb_audio.shape[1]), "normalization": "catalog z-score, L2 row"},
 "n_songs": len(songs), "failures": failures,
}
(DATA_DIR / "meta.json").write_text(json.dumps(meta, indent=1))
print(f"wrote {DATA_DIR}: {len(songs)} songs | brain {emb_brain.shape} | audio {emb_audio.shape} | failures {len(failures)}")
assert emb_brain.shape[0] == emb_audio.shape[0] == len(songs) == len(list(tl_dir.glob('*.npy')))

In [ ]:
# [C8] Summary + eyeball check: neighbors for 3 random songs in both spaces, and catalog-wide overlap
log = [json.loads(l) for l in open(LOG)] if LOG.exists() else []
ok = [e for e in log if e["ok"]]
print(f"songs in data/: {len(songs)} / {len(songs_meta)} audio files | failures: {len(meta['failures'])}")
if ok: print(f"compute: {sum(e['seconds'] for e in ok)/60:.1f} min for {sum(e['audio_s'] for e in ok)/60:.0f} min of audio "
             f"({sum(e['seconds'] for e in ok)/(sum(e['audio_s'] for e in ok)/60):.1f} s per audio-minute)")
for f in meta["failures"]: print("  FAILED:", f["filename"], "->", f["error"][:120])

def nn(emb, i, k=5):
    s = emb @ emb[i]; s[i] = -np.inf; o = np.argsort(-s)[:k]; return [(int(j), float(s[j])) for j in o]
lab = lambda j: f"{songs[j]['artist'] or ''} - {songs[j]['title']}"[:48]
rng = np.random.default_rng(0)
for i in rng.choice(len(songs), size=min(3, len(songs)), replace=False):
    print(f"\n== {lab(i)}")
    for (jb, sb), (ja, sa) in zip(nn(emb_brain, i), nn(emb_audio, i)):
        print(f"   brain {sb:+.3f} {lab(jb):48s} | audio {sa:+.3f} {lab(ja)}")

K = 10; jac = []
for i in range(len(songs)):
    b = {j for j, _ in nn(emb_brain, i, K)}; a = {j for j, _ in nn(emb_audio, i, K)}; jac.append(len(a & b) / len(a | b))
print(f"\ncatalog mean overlap@{K}: Jaccard {np.mean(jac):.3f} (random lists would give ~{K/(2*(len(songs)-1)-K):.3f})")
net_means = np.stack([t.mean(0) @ NETMAT for t in timelines])
print("catalog mean per network:", dict(zip(NET_ORDER, net_means.mean(0).round(3))))
shutil.make_archive(str(OUT_DIR / "spotimind_data"), "zip", DATA_DIR)
print("zipped ->", OUT_DIR / "spotimind_data.zip", f"{(OUT_DIR / 'spotimind_data.zip').stat().st_size/1e6:.1f} MB")

In [ ]:
# [C9] Download data/ to this computer (lands in the browser's Downloads folder).
# Then, in the repo:  unzip -o ~/Downloads/spotimind_data.zip -d data
from google.colab import files
files.download(str(OUT_DIR / "spotimind_data.zip"))